# 04 — Feature Engineering

## Objective

The objective of this notebook is to create meaningful time-based and historical features for electricity consumption prediction. These features will help machine learning models capture daily, weekly, and longer-term consumption patterns while avoiding data leakage.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "household_power_consumption_cleaned.csv"
)

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Datetime"],
    index_col="Datetime"
)

print("Dataset shape:", df.shape)
print("Datetime index:", isinstance(df.index, pd.DatetimeIndex))
print("Chronological order:", df.index.is_monotonic_increasing)

display(df.head())

Dataset shape: (2075259, 7)
Datetime index: True
Chronological order: True


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


This cell loads the cleaned dataset and verifies its dimensions, datetime index, and chronological order before feature engineering begins.

In [3]:
df_features = df.copy()

df_features["Hour"] = df_features.index.hour
df_features["DayOfWeek"] = df_features.index.day_of_week
df_features["Month"] = df_features.index.month
df_features["DayOfMonth"] = df_features.index.day
df_features["IsWeekend"] = (
    df_features["DayOfWeek"] >= 5
).astype(int)

print("Dataset shape after feature engineering:", df_features.shape)

display(
    df_features[
        [
            "Hour",
            "DayOfWeek",
            "Month",
            "DayOfMonth",
            "IsWeekend"
        ]
    ].head()
)

Dataset shape after feature engineering: (2075259, 12)


,Hour,DayOfWeek,Month,DayOfMonth,IsWeekend
Datetime,,,,,
2006-12-16 17:24:00,17,5,12,16,1
2006-12-16 17:25:00,17,5,12,16,1
2006-12-16 17:26:00,17,5,12,16,1
2006-12-16 17:27:00,17,5,12,16,1
2006-12-16 17:28:00,17,5,12,16,1


This cell creates time-based features from the Datetime index: hour of the day, day of the week, month, day of the month, and a weekend indicator. These features may help machine learning models capture recurring electricity consumption patterns.

In [4]:
target_column = "Global_active_power"

df_features["Power_Lag_1"] = (
    df_features[target_column].shift(1)
)

df_features["Power_Lag_60"] = (
    df_features[target_column].shift(60)
)

df_features["Power_Lag_1440"] = (
    df_features[target_column].shift(1440)
)

display(
    df_features[
        [
            target_column,
            "Power_Lag_1",
            "Power_Lag_60",
            "Power_Lag_1440"
        ]
    ].head(5)
)

print("Dataset shape:", df_features.shape)


,Global_active_power,Power_Lag_1,Power_Lag_60,Power_Lag_1440
Datetime,,,,
2006-12-16 17:24:00,4.216,NaN,NaN,NaN
2006-12-16 17:25:00,5.360,4.216,NaN,NaN
2006-12-16 17:26:00,5.374,5.360,NaN,NaN
2006-12-16 17:27:00,5.388,5.374,NaN,NaN
2006-12-16 17:28:00,3.666,5.388,NaN,NaN


Dataset shape: (2075259, 15)


This cell creates lag features from Global Active Power. Power_Lag_1 represents consumption one minute earlier, Power_Lag_60 represents consumption 60 minutes earlier, and Power_Lag_1440 represents consumption 1,440 minutes earlier (one day, assuming regular one-minute sampling).

In [5]:
df_features["Power_Rolling_Mean_60"] = (
    df_features["Global_active_power"]
    .shift(1)
    .rolling(window=60, min_periods=60)
    .mean()
)

df_features["Power_Rolling_Mean_1440"] = (
    df_features["Global_active_power"]
    .shift(1)
    .rolling(window=1440, min_periods=1440)
    .mean()
)

display(
    df_features[
        [
            "Global_active_power",
            "Power_Rolling_Mean_60",
            "Power_Rolling_Mean_1440"
        ]
    ].head(10)
)

print("Dataset shape:", df_features.shape)


,Global_active_power,Power_Rolling_Mean_60,Power_Rolling_Mean_1440
Datetime,,,
2006-12-16 17:24:00,4.216,NaN,NaN
2006-12-16 17:25:00,5.360,NaN,NaN
2006-12-16 17:26:00,5.374,NaN,NaN
2006-12-16 17:27:00,5.388,NaN,NaN
2006-12-16 17:28:00,3.666,NaN,NaN
2006-12-16 17:29:00,3.520,NaN,NaN
2006-12-16 17:30:00,3.702,NaN,NaN
2006-12-16 17:31:00,3.700,NaN,NaN
2006-12-16 17:32:00,3.668,NaN,NaN


Dataset shape: (2075259, 17)


This cell creates rolling mean features using the previous 60 minutes and previous 1,440 minutes of Global Active Power. Shifting the target by one row ensures that each rolling mean uses past observations only, preventing the current target value from entering its own feature.

In [6]:
rolling_columns = [
    "Power_Rolling_Mean_60",
    "Power_Rolling_Mean_1440"
]

print("Missing values in rolling features:")
display(df_features[rolling_columns].isna().sum())

print("\nNon-missing values in rolling features:")
display(df_features[rolling_columns].notna().sum())


Missing values in rolling features:


Power_Rolling_Mean_60      26857
Power_Rolling_Mean_1440    48937
dtype: int64


Non-missing values in rolling features:


Power_Rolling_Mean_60      2048402
Power_Rolling_Mean_1440    2026322
dtype: int64

This cell checks the number of missing and non-missing values in the rolling mean features. It helps verify how much historical information is available for subsequent modeling.

In [7]:
lag_columns = [
    "Power_Lag_1",
    "Power_Lag_60",
    "Power_Lag_1440"
]

print("Missing values in lag features:")
display(df_features[lag_columns].isna().sum())

print("\nNon-missing values in lag features:")
display(df_features[lag_columns].notna().sum())

Missing values in lag features:


Power_Lag_1       25904
Power_Lag_60      25963
Power_Lag_1440    27343
dtype: int64


Non-missing values in lag features:


Power_Lag_1       2049355
Power_Lag_60      2049296
Power_Lag_1440    2047916
dtype: int64

This cell counts missing and non-missing values in the lag features. The results help assess how much historical information is available for each lag before preparing the final modeling dataset.

In [8]:
feature_columns = [
    "Hour",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "Power_Lag_1",
    "Power_Lag_60",
    "Power_Lag_1440",
    "Power_Rolling_Mean_60",
    "Power_Rolling_Mean_1440"
]

print("Original columns:", df.shape[1])
print("Engineered features:", len(feature_columns))
print("Current dataset shape:", df_features.shape)

print("\nFeature data types:")
display(df_features[feature_columns].dtypes)

print("\nFeature preview:")
display(df_features[feature_columns].head())

Original columns: 7
Engineered features: 10
Current dataset shape: (2075259, 17)

Feature data types:


Hour                         int32
DayOfWeek                    int32
Month                        int32
DayOfMonth                   int32
IsWeekend                    int64
Power_Lag_1                float64
Power_Lag_60               float64
Power_Lag_1440             float64
Power_Rolling_Mean_60      float64
Power_Rolling_Mean_1440    float64
dtype: object


Feature preview:


,Hour,DayOfWeek,Month,DayOfMonth,IsWeekend,Power_Lag_1,Power_Lag_60,Power_Lag_1440,Power_Rolling_Mean_60,Power_Rolling_Mean_1440
Datetime,,,,,,,,,,
2006-12-16 17:24:00,17,5,12,16,1,NaN,NaN,NaN,NaN,NaN
2006-12-16 17:25:00,17,5,12,16,1,4.216,NaN,NaN,NaN,NaN
2006-12-16 17:26:00,17,5,12,16,1,5.360,NaN,NaN,NaN,NaN
2006-12-16 17:27:00,17,5,12,16,1,5.374,NaN,NaN,NaN,NaN
2006-12-16 17:28:00,17,5,12,16,1,5.388,NaN,NaN,NaN,NaN


In [9]:
print("Time differences:")
display(df_features.index.to_series().diff().value_counts().head())

print("\nMissing target values:")
print(df_features["Global_active_power"].isna().sum())

print("\nMissing values in all engineered features:")
display(df_features[feature_columns].isna().sum())

Time differences:


Datetime
0 days 00:01:00    2075258
Name: count, dtype: int64


Missing target values:
25903

Missing values in all engineered features:


Hour                           0
DayOfWeek                      0
Month                          0
DayOfMonth                     0
IsWeekend                      0
Power_Lag_1                25904
Power_Lag_60               25963
Power_Lag_1440             27343
Power_Rolling_Mean_60      26857
Power_Rolling_Mean_1440    48937
dtype: int64

This cell checks the time intervals between consecutive observations, counts missing target values, and summarizes missing values across all engineered features. These checks help identify data-quality issues before preparing the modeling dataset.

In [10]:
required_features = [
    "Hour",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "Power_Lag_1",
    "Power_Lag_60",
    "Power_Lag_1440",
    "Power_Rolling_Mean_60",
    "Power_Rolling_Mean_1440"
]

complete_feature_mask = df_features[required_features].notna().all(axis=1)

print("Total rows:", len(df_features))
print("Rows with all features available:", complete_feature_mask.sum())
print("Rows with at least one missing feature:", (~complete_feature_mask).sum())

Total rows: 2075259
Rows with all features available: 2026322
Rows with at least one missing feature: 48937


This cell identifies rows where all engineered features are available without modifying or deleting any data. It estimates how many observations could be used for a model that requires every listed feature.

In [11]:
target_column = "Global_active_power"

model_columns = feature_columns + [target_column]

model_df = df_features[model_columns].dropna().copy()

print("Original rows:", len(df_features))
print("Rows available for modeling:", len(model_df))
print("Columns in modeling dataset:", model_df.shape[1])
print("Remaining missing values:", model_df.isna().sum().sum())

display(model_df.head())

Original rows: 2075259
Rows available for modeling: 2026307
Columns in modeling dataset: 11
Remaining missing values: 0


,Hour,DayOfWeek,Month,DayOfMonth,IsWeekend,Power_Lag_1,Power_Lag_60,Power_Lag_1440,Power_Rolling_Mean_60,Power_Rolling_Mean_1440,Global_active_power
Datetime,,,,,,,,,,,
2006-12-17 17:24:00,17,6,12,17,1,3.238,4.864,4.216,3.552533,2.487728,3.390
2006-12-17 17:25:00,17,6,12,17,1,3.390,4.868,5.360,3.527967,2.487154,3.360
2006-12-17 17:26:00,17,6,12,17,1,3.360,4.876,5.374,3.502833,2.485765,3.400
2006-12-17 17:27:00,17,6,12,17,1,3.400,4.810,5.388,3.478233,2.484394,3.396
2006-12-17 17:28:00,17,6,12,17,1,3.396,4.790,3.666,3.454667,2.483011,3.282


This cell creates a separate modeling dataset containing the selected engineered features and the target variable. Rows with missing values in any selected feature or the target are removed from this modeling copy, while the original engineered dataset remains unchanged.

In [12]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "energy_modeling_dataset.csv"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=True,
    index_label="Datetime"
)

print("Saved dataset to:", OUTPUT_PATH)
print("Final shape:", model_df.shape)
print("Missing values:", model_df.isna().sum().sum())

Saved dataset to: c:\Users\hp\smart-energy-consumption-prediction\data\processed\energy_modeling_dataset.csv
Final shape: (2026307, 11)
Missing values: 0


This cell saves the prepared modeling dataset as a CSV file, preserving Datetime as a column. It then verifies the output path, dataset dimensions, and absence of missing values.

In [13]:
time_diff = model_df.index.to_series().diff()

print("Most common time intervals:")
display(time_diff.value_counts().head(10))

print("\nIntervals different from one minute:")
print((time_diff.dropna() != pd.Timedelta(minutes=1)).sum())

Most common time intervals:


Datetime
0 days 00:01:00    2026291
3 days 14:04:00          1
1 days 00:34:00          1
1 days 02:20:00          1
1 days 00:22:00          1
1 days 00:44:00          1
1 days 00:07:00          1
1 days 01:11:00          1
1 days 00:39:00          1
1 days 00:25:00          1
Name: count, dtype: int64


Intervals different from one minute:
15


This cell checks the time intervals between consecutive rows in the modeling dataset. It identifies whether removing incomplete observations has introduced gaps between retained timestamps.


### Temporal Consistency Check

The modeling dataset contains 2,026,307 rows with no missing values in the selected features or target. Most consecutive observations are one minute apart, but 15 intervals exceed one minute because incomplete rows were removed.

Therefore, lag features represent previous retained observations and may not always correspond to the exact intended time offset across gaps. This limitation must be considered during time-series model evaluation.

In [15]:
# Create a copy of the target series
power = df_features["Global_active_power"].copy()

# Create time-based lag features
df_features["Power_Lag_1"] = power.shift(freq="1min")
df_features["Power_Lag_60"] = power.shift(freq="60min")
df_features["Power_Lag_1440"] = power.shift(freq="1440min")

# Verify the new lag features
display(
    df_features[
        [
            "Global_active_power",
            "Power_Lag_1",
            "Power_Lag_60",
            "Power_Lag_1440"
        ]
    ].head()
)

print("Missing values in corrected lag features:")
display(
    df_features[
        ["Power_Lag_1", "Power_Lag_60", "Power_Lag_1440"]
    ].isna().sum()
)

,Global_active_power,Power_Lag_1,Power_Lag_60,Power_Lag_1440
Datetime,,,,
2006-12-16 17:24:00,4.216,NaN,NaN,NaN
2006-12-16 17:25:00,5.360,4.216,NaN,NaN
2006-12-16 17:26:00,5.374,5.360,NaN,NaN
2006-12-16 17:27:00,5.388,5.374,NaN,NaN
2006-12-16 17:28:00,3.666,5.388,NaN,NaN


Missing values in corrected lag features:


Power_Lag_1       25904
Power_Lag_60      25963
Power_Lag_1440    27343
dtype: int64

This cell recreates lag features using actual time offsets rather than row positions. The features represent Global Active Power one minute, 60 minutes, and 1,440 minutes earlier, respectively. Missing values remain where the exact historical timestamp is unavailable.

In [16]:
power = df_features["Global_active_power"]

df_features["Power_Rolling_Mean_60"] = (
    power.shift(freq="1min")
    .rolling("60min", min_periods=60)
    .mean()
)

df_features["Power_Rolling_Mean_1440"] = (
    power.shift(freq="1min")
    .rolling("1440min", min_periods=1440)
    .mean()
)

display(
    df_features[
        [
            "Global_active_power",
            "Power_Rolling_Mean_60",
            "Power_Rolling_Mean_1440"
        ]
    ].head(10)
)

print("Missing values in corrected rolling features:")
display(
    df_features[
        ["Power_Rolling_Mean_60", "Power_Rolling_Mean_1440"]
    ].isna().sum()
)

,Global_active_power,Power_Rolling_Mean_60,Power_Rolling_Mean_1440
Datetime,,,
2006-12-16 17:24:00,4.216,NaN,NaN
2006-12-16 17:25:00,5.360,NaN,NaN
2006-12-16 17:26:00,5.374,NaN,NaN
2006-12-16 17:27:00,5.388,NaN,NaN
2006-12-16 17:28:00,3.666,NaN,NaN
2006-12-16 17:29:00,3.520,NaN,NaN
2006-12-16 17:30:00,3.702,NaN,NaN
2006-12-16 17:31:00,3.700,NaN,NaN
2006-12-16 17:32:00,3.668,NaN,NaN


Missing values in corrected rolling features:


Power_Rolling_Mean_60      26857
Power_Rolling_Mean_1440    48937
dtype: int64

This cell recalculates rolling mean features using time-based windows. The one-minute shift excludes the current target observation, while the 60-minute and 1,440-minute windows require a complete set of historical measurements.

In [17]:
model_columns = feature_columns + ["Global_active_power"]

model_df = df_features[model_columns].dropna().copy()

print("Original rows:", len(df_features))
print("Rows available for modeling:", len(model_df))
print("Dataset shape:", model_df.shape)
print("Remaining missing values:", model_df.isna().sum().sum())

Original rows: 2075259
Rows available for modeling: 2026307
Dataset shape: (2026307, 11)
Remaining missing values: 0


This cell rebuilds the modeling dataset using the updated time-based lag and rolling features. Rows with missing values in the selected features or target are excluded from the modeling copy, while the full engineered dataset remains unchanged.

In [18]:
power = df_features["Global_active_power"]

previous_60 = power.shift(freq="1min").rolling(
    "60min", min_periods=60
)

print("Rolling feature check completed.")
print("60-minute rolling mean missing:", df_features["Power_Rolling_Mean_60"].isna().sum())
print("1440-minute rolling mean missing:", df_features["Power_Rolling_Mean_1440"].isna().sum())

Rolling feature check completed.
60-minute rolling mean missing: 26857
1440-minute rolling mean missing: 48937


This cell performs a final count check on the rolling mean features after their time-based recalculation. It confirms the current number of missing rolling values before the modeling dataset is saved.

In [19]:
power = df_features["Global_active_power"]

valid_60 = (
    power.notna()
    .rolling(window=60, min_periods=60)
    .sum()
    .shift(1)
    .eq(60)
)

valid_1440 = (
    power.notna()
    .rolling(window=1440, min_periods=1440)
    .sum()
    .shift(1)
    .eq(1440)
)

print("Rows with 60 valid previous measurements:", valid_60.sum())
print("Rows with 1440 valid previous measurements:", valid_1440.sum())

print(
    "60-minute rolling values with complete history:",
    (df_features["Power_Rolling_Mean_60"].notna() & valid_60).sum()
)

print(
    "1440-minute rolling values with complete history:",
    (df_features["Power_Rolling_Mean_1440"].notna() & valid_1440).sum()
)

Rows with 60 valid previous measurements: 2048402
Rows with 1440 valid previous measurements: 2026322
60-minute rolling values with complete history: 2048402
1440-minute rolling values with complete history: 2026322


This cell checks whether the previous 60 or 1,440 consecutive rows contain complete Global Active Power measurements. It compares these validity masks with the existing rolling features to identify windows that have sufficient historical data.

In [20]:
model_columns = feature_columns + ["Global_active_power"]

model_df = df_features[model_columns].dropna().copy()

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "energy_modeling_dataset.csv"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=True,
    index_label="Datetime"
)

print("Saved to:", OUTPUT_PATH)
print("Final dataset shape:", model_df.shape)
print("Missing values:", model_df.isna().sum().sum())
print(
    "Time intervals different from one minute:",
    (
        model_df.index.to_series()
        .diff()
        .dropna()
        .ne(pd.Timedelta(minutes=1))
    ).sum()
)

Saved to: c:\Users\hp\smart-energy-consumption-prediction\data\processed\energy_modeling_dataset.csv
Final dataset shape: (2026307, 11)
Missing values: 0
Time intervals different from one minute: 15


This cell rebuilds and saves the modeling dataset using the corrected time-based lag and rolling features. It removes rows with missing values in the selected features or target, then verifies the saved dataset dimensions, missing-value count, and remaining time gaps.

# Conclusion

This notebook introduced time-based and historical features for electricity consumption prediction.

The engineered features include hour of day, day of week, month, day of month, weekend indicator, and historical Global Active Power measurements. Lag features were defined using actual time offsets, while rolling means were calculated using previous observations only.

A modeling dataset containing 2,026,307 rows and 11 columns was saved to `data/processed/energy_modeling_dataset.csv`. The saved dataset contains no missing values.

After removing incomplete observations, 15 intervals differ from one minute. This temporal irregularity must be considered during model evaluation to ensure that training and testing respect the chronological order of the observations.

The dataset is now prepared for the next stage: chronological train-test splitting and predictive modeling.